In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

model_name = "t5-small"
output_dir = "./t5_text2sql_model"

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_data():
    data = load_dataset("json", data_files={
        "train": "data/wikisql/wikisql_train.json",
        "validation": "data/wikisql/wikisql_test.json",
    })

    data["train"] = data["train"].shuffle(seed= 42).select(range(10000))
    data["validation"] = data["validation"].shuffle(seed= 42).select(range(1000))

    return data

dataset = load_data()

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [6]:
prefix = "translate English to SQL: "

def preprocess(examples):
    inputs = [prefix + q for q in examples["question"]]
    targets = examples["answer"]

    model_inputs = tokenizer(inputs, max_length=256, truncation=True)
    labels = tokenizer(text_target=targets, max_length=256, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy= "epoch",
    save_strategy= "epoch",
    learning_rate= 3e-5,
    per_device_train_batch_size= 8,
    per_device_eval_batch_size= 8,
    num_train_epochs= 3,
    predict_with_generate= True,
    fp16=torch.cuda.is_available(),
    report_to= "none"
)

trainer = Seq2SeqTrainer(
    model= model,
    args= args,
    train_dataset= tokenized["train"],
    eval_dataset= tokenized["validation"],
    processing_class= tokenizer,
    data_collator= DataCollatorForSeq2Seq(tokenizer, model)
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.037972,0.746178
2,0.796229,0.663758
3,0.775751,0.642631


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.9595259847005209, metrics={'train_runtime': 1922.1713, 'train_samples_per_second': 15.607, 'train_steps_per_second': 1.951, 'total_flos': 275033573031936.0, 'train_loss': 0.9595259847005209, 'epoch': 3.0})

In [10]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./t5_text2sql_model\\tokenizer_config.json',
 './t5_text2sql_model\\tokenizer.json')

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "./t5_text2sql_model"

def generate_sql(question, model_path= MODEL_PATH):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

    model.eval()

    prefix = "translate English to SQL: "
    input_text = prefix + question

    inputs = tokenizer(input_text, return_tensors= "pt", truncation= True, max_length= 256).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length= 256, num_beams= 4, early_stopping= True)

    sql_query = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return sql_query



sample_question = "How many schools did player number 3 play at?"
result = generate_sql(sample_question)

print("\nQuestion:", sample_question)
print("SQL:", result)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


Question: How many schools did player number 3 play at?
SQL: SELECT COUNT Schools FROM table WHERE Player = 3


In [14]:
generate_sql("how many students that them age bigger than 20 and gpa less or equal 3")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

'SELECT COUNT Student FROM table WHERE Age > 20 AND GPA  3'